# Where-fi - Object localization using RSSI of Wi-fi signals indoors

### Project for the INTAR lecture at UPV

## Imports and configurations

### Imports of useful libraries

In [ ]:
!pip install ax-platform
!pip install torchmetrics

In [22]:
# Libraries for logging and general utilities
import random
import logging
import wandb
import time
import math
import yaml
from box.box import Box
import functools
import operator
import gc

# Libraries for data manipulation and analysis
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from ax.service.ax_client import AxClient, ObjectiveProperties

# Helpful libraries
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional, Tuple, Union

# Deep learning libraries
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchmetrics.classification import (MulticlassAccuracy, MulticlassConfusionMatrix,
                                         MulticlassF1Score, MulticlassAUROC)
from torchmetrics.regression import (MeanAbsoluteError,
                                     MeanSquaredError)
from torchmetrics.functional.pairwise import pairwise_euclidean_distance

### Configuration to control model architecture and training

In [23]:
%%writefile Ensemble_MLP.yaml
name: "Ensemble_MLP"
classifier:
    input_dim: 520
    dropout: 0.4426632479755182
    floor_classes: 5
regressor:
    input_dim: 520
    dropout: 0.12498379261035723

loss_weights:
    classification: 0.001
    regression: 1.0

training:
    num_epochs: 75
    # Learning rate parameters
    learning_rate: 0.0003
    learning_rate_scheduler: "cosine"
    learning_rate_min: 0.000001
    # Optimizer parameters
    beta1: 0.9
    beta2: 0.999
    weight_decay: 0.01

Overwriting Ensemble_MLP.yaml


In [24]:
%%writefile Autoencoder.yaml
name: "Autoencoder"
input_dim: 520
cnn_in_dim: 40
autoencoder:
    # Encoder and decoder dimensions correct that larger than input_dim, 
    # see paper "Indoor Localization With an Autoencoder-Based Convolutional 
    # Neural Network"
    fc_enc_dim: 2048
    fc_dec_dim: 2048
    freeze_ae_training: 0.0
classifier:
    dropout: 0.3
    floor_classes: 5
regressor:
    dropout: 0.3
loss_weights:
    classification: 0.05
    regression: 2
    decoder: 3

training:
    num_epochs: 220
    # Learning rate parameters
    learning_rate: 0.0003
    learning_rate_scheduler: "cosine"
    learning_rate_min: 0.000001
    # Optimizer parameters
    beta1: 0.9
    beta2: 0.999
    weight_decay: 0.01

Overwriting Autoencoder.yaml


In [ ]:
%%writefile config.yaml
# Change 'local' to 'colab' if running in Google Colab, don't forget to adapt source path
environment: 'colab'
# Seed for deterministics
seed: 42

# Number of classes for floor classification
floor_classes: 5
logging:
    log_interval: 5
    eval_interval: 5
    # For WandB logging
    use_wandb: False
    team_name: "UPV_projects"
    project_name: "Where-fi"
    api_key: null
    # Add-on for the run name
    run_name_additional: 'training'
use_model: "Autoencoder"
#use_model: "Autoencoder", "Ensemble_MLP"
dataset:
    batch_size: 64
    # For loading
    root_path: '../Dataset/UJIndoorLoc'
    training_data_name: "trainingData.csv"
    validation_data_name: "validationData.csv"
    # For normalization & preprocessing
    no_signal_value: -110
    signal_max: 0
    longitude_min: -7695.9387549299299000
    longitude_max: -7299.786516730871000
    latitude_min: 4864745.7450159714
    latitude_max: 4865017.3646842018
    # For dataaugmentation
    data_augmentation:
        noise:
            enabled: True,
            noise_level: 0.091
            probability: 0.75
        dropout:
            enabled: True,
            probability: 0.05
        shift:
            enabled: True,
            max_shift: 0.0455
            probability: 0.6

Overwriting config.yaml


In [26]:
%%writefile optimization_config.yaml
# Main switch to enable or disable the optimization process
active: False

# Enable Weights & Biases logging for optimization trials
wb_logging: False

# Configuration specifying the objectives the optimizer should try to achieve
optimization_config:
  objectives:
    classification_accuracy:
      minimize: False    # We want to maximize accuracy
      threshold: 0.90    # Optional: Min tolerated accuracy
    regression_mse:
      minimize: True     # We want to minimize Mean Squared Error
      threshold: 0.001   # Optional: Max tolerated MSE
  # Total number of trial runs for the Bayesian optimization
  total_trials: 200

# Defines the hyperparameters to tune and their respective bounds
search_space:
  # Dropout rate for the classifier head
  - name: "model.classifier.dropout"
    type: "range"
    bounds: [0.0, 0.8]
    value_type: "float"
  
  # Dropout rate for the regressor head
  - name: "model.regressor.dropout"
    type: "range"
    bounds: [0.0, 0.8]
    value_type: "float"

  # Number of epochs for training each trial
  - name: "training.num_epochs"
    type: "range"
    bounds: [20, 250]
    value_type: "int"
  
  # Max learning rate for the Cosine Annealing scheduler (or general LR)
  - name: "training.learning_rate"
    type: "range"
    bounds: [0.00001, 0.01]
    value_type: "float"
  
  # Min learning rate for the Cosine Annealing scheduler
  - name: "training.learning_rate_min"
    type: "range"
    bounds: [0.0000001, 0.001]
    value_type: "float"

  # Weight decay (L2 regularization) coefficient
  - name: "training.weight_decay"
    type: "range"
    bounds: [0.01, 0.5]
    value_type: "float"

  # Batch size for DataLoader
  - name: "dataset.batch_size"
    type: "range"
    bounds: [32, 128]
    value_type: "int"

Overwriting optimization_config.yaml


In [27]:
# Load general configuration using Box to allow dot-notation access (e.g., CONFIG.training.batch_size)
with open("config.yaml", "r") as f:
    CONFIG = Box(yaml.safe_load(f))
print(f"General configuration loaded from config.yaml")

# Load model-specific configuration dynamically based on the 'use_model' specified in config.yaml
with open(f"{CONFIG.use_model}.yaml", "r") as f:
    MODEL_CONFIG = Box(yaml.safe_load(f))
print(f"Model-specific configuration loaded for model {MODEL_CONFIG.name}")

# Merge the training parameters from the model config into the main config
# This allows models to define their own specific training parameters (e.g., epochs, learning rate)
CONFIG.training = MODEL_CONFIG.training

# Remove the duplicated training section from MODEL_CONFIG to keep structure clean
del MODEL_CONFIG.training

# Embed the remaining model-specific settings (like architecture hyperparams) under 'model'
CONFIG.model = MODEL_CONFIG
print(f"Configuration merged for model {CONFIG.model.name}")

# Load hyperparameter optimization configuration and embed it under 'optimization'
with open(f"optimization_config.yaml", "r") as f:
    OPTIMIZATION_CONFIG = Box(yaml.safe_load(f))
    CONFIG.optimization = OPTIMIZATION_CONFIG
print(f"Optimization configuration loaded")

General configuration loaded from config.yaml
Model-specific configuration loaded for model Ensemble_MLP
Configuration merged for model Ensemble_MLP
Optimization configuration loaded


#### Support of Google Colab

In [28]:
# Handle Google Colab specific environment setup
if CONFIG.environment == 'colab':
    # Mount Google Drive to access the dataset stored in the user's Drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    
    # Override the default local dataset path with the Google Drive path
    dataset_path = 'gdrive/MyDrive/Colab Notebooks/where_fi/dataset'
    CONFIG.dataset.root_path = dataset_path

# Handle local execution (e.g. running on local machine)
elif CONFIG.environment == 'local':
    # No additional setup needed, uses default root_path from config.yaml
    pass

# Catch unsupported environment values
else:
  raise ValueError(f"Invalid environment '{CONFIG.environment}'. Must be one of 'local' or 'colab'.")

## Helper classes
Classes for utility functionalities

### Class for training parameters
Small helper class to organise the parameters for the training

In [29]:
class TrainingParameters:
    '''
    Class to encapsulate and manage training parameters.
    It holds optimization hyperparameters such as learning rate, beta values for Adam/AdamW,
    weight decay for regularization, the chosen learning rate scheduler strategy, 
    and the total number of epochs. 
    Using this class makes passing these arguments to the optimizer and trainer cleaner.
    '''

    def __init__(self,
            lr: float,
            lr_min: float,
            lr_scheduler: str,
            beta1: float,
            beta2: float,
            weight_decay: float,
            epochs: int
        ):
        '''
        Initializes the training parameters with the given values.
        
        Args:
            lr (float): The maximum/starting learning rate.
            lr_min (float): The minimum learning rate (used by schedulers like cosine annealing).
            lr_scheduler (str): The name/type of learning rate scheduler to use (e.g., 'cosine').
            beta1 (float): Beta1 coefficient for computing running averages of gradient in optimizer.
            beta2 (float): Beta2 coefficient for computing running averages of squared gradient.
            weight_decay (float): L2 penalty factor (weight decay) applied to the optimizer.
            epochs (int): Total number of training epochs to run.
        '''
        self.lr = lr
        self.lr_min = lr_min
        self.lr_scheduler = lr_scheduler
        self.beta1 = beta1
        self.beta2 = beta2
        self.weight_decay = weight_decay
        self.epochs = epochs

    def get_learning_rate(self) -> float:
        '''
        Get the configured maximum/initial learning rate.
        '''
        return self.lr

    def get_learning_rate_min(self) -> float:
        '''
        Get the configured minimum learning rate for the scheduler.
        '''
        return self.lr_min

    def get_betas(self) -> tuple[float, float]:
        '''
        Get the beta values as a tuple, required format for configuring PyTorch optimizers like Adam.
        '''
        return (self.beta1, self.beta2)

    def get_weight_decay(self) -> float:
        '''
        Get the weight decay (L2 regularization) coefficient.
        '''
        return self.weight_decay

    def get_epochs(self) -> int:
        '''
        Get the total number of epochs to train.
        '''
        return self.epochs

    def get_lr_scheduler(self) -> str:
        '''
        Get the name identifier for the requested learning rate scheduler.
        '''
        return self.lr_scheduler

## Dataset

### Dataaugmentation

In [30]:
class GaussianNoise:
    '''
    Data augmentation transformation that adds random Gaussian noise to the input signal (RSSI values).
    This helps the model to effectively handle natural fluctuations and noise present in real-world environments, 
    preventing overfitting to the exact training distributions.
    '''
    def __init__(self, std_range=(0.0, 0.1), p=0.0):
        """
        Args:
            std_range (tuple): Range of standard deviations to randomly choose from. 
                               Controls the magnitude of the applied noise.
            p (float): Probability (between 0.0 and 1.0) that the noise augmentation 
                       will be applied to a given sample during training.
        """
        self.std_range = std_range
        self.p = p

    def __call__(self, x):
        """
        Applies the noise operation to the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (Batch_Size, Features).
        
        Returns:
            torch.Tensor: Augmented tensor with noise added (if triggered by probability), 
                          or the original tensor unaffected.
        """
        # Apply noise only if the random roll is below the configured probability (p)
        if random.random() < self.p:
            # Randomly select a standard deviation within the allowed range
            std = random.uniform(self.std_range[0], self.std_range[1])
            
            # Generate random Gaussian noise with the same shape as the input tensor (x)
            noise = torch.randn_like(x) * std
            
            # Limit the absolute maximal influence of the noise to prevent 
            # completely distorting/destroying the original signal pattern
            noise = torch.clamp(noise, -self.std_range[1], self.std_range[1])
            
            # Add the generated noise to the original signal
            result = x + noise
            
            # Ensure the final result stays strictly within the normalized bounds of [0.0, 1.0]
            result = torch.clamp(result, 0.0, 1.0)
            
            return result
            
        # Return unmodified input if the random check failed
        return x

In [31]:
class RandomValueDropout:
    '''
    Data augmentation transformation that simulates signal loss or sensor failure.
    Randomly drops (sets to zero) some features within an input vector. 
    This forces the model to learn from partial data and not rely too heavily on any single WiFi AP.
    '''
    def __init__(self, p=0.0):
        '''
        Args:
            p (float): Global probability (between 0.0 and 1.0) that this dropout augmentation 
                       will be triggered for a given batch.
        '''
        self.p = p

    def __call__(self, x):
        '''
        Applies the random dropout operation to the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (Batch_Size, Features).
        
        Returns:
            torch.Tensor: Augmented tensor with ~10% of elements zeroed out (if triggered), 
                          or the original tensor unaffected.
        '''
        # Only apply dropout if the random check passes based on probability 'p'
        if random.random() < self.p:
            # Create a boolean mask of the same shape as 'x'.
            # torch.rand_like generates values between [0, 1). > 0.9 means roughly 10% of values will be True.
            mask = torch.rand_like(x) > 0.9
            
            # Fill the True elements in the mask with 0.0 (simulating dropped RSSI signals)
            x = x.masked_fill(mask, 0.0)
            
        return x

In [32]:
class Shifting:
    '''
    Data augmentation transformation that applies a uniform bias shift to all active RSSI signals dynamically.
    This simulates environmental changes (e.g., increased humidity, differently calibrated phone antennas, 
    or temporary overall network interference) that shift the entire signal baseline up or down.
    '''
    def __init__(self, shift=0.05, p=0.0):
        """
        Args:
            shift (float): Maximum absolute bounds for the random shift. 
                           Since data is normalized between 0.0 and 1.0, a shift of 0.05 targets ~5% change.
            p (float): Global probability (between 0.0 and 1.0) that this shift augmentation 
                       will be applied to a given batch during training.
        """
        self.shift = shift
        self.p = p

    def __call__(self, x):
        """
        Applies a random scalar shift broadcast across the entire input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (Batch_Size, Features).
        
        Returns:
            torch.Tensor: Augmented tensor with a uniform shift added (if triggered), 
                          or the original tensor unaffected.
        """
        # Apply shift only if the random trigger falls within defined probability
        if random.random() < self.p:
            # Generate a single random float value between the negative and positive shift bounds
            # For example, if shift bounds are 0.05, it could generate -0.02, raising/lowering the baseline universally.
            shift = random.uniform(-self.shift, self.shift)
            
            # Broadcast add the scalar shift to every element in the tensor array simultaneously
            result = x + shift
            
            # Ensure the mutated values don't break normalization rules by clamping them
            # strictly back into the [0.0, 1.0] range. (e.g. 0.98 + 0.05 = 1.03 -> clamped to 1.0)
            result = torch.clamp(result, 0.0, 1.0)
            
            return result
            
        return x

In [33]:
class DataAugmentation:
    """
    A composable data augmentation pipeline for Wi-Fi RSSI signals.
    
    This class wraps multiple individual transformations (Gaussian noise, 
    uniform baseline shifting, and random access point dropout) into a 
    single callable pipeline using `torchvision.transforms.Compose`. 
    
    In real-world Wi-Fi localization, these augmentations help make the 
    neural network robust against:
      - Sensor measurement noise (GaussianNoise)
      - Environmental condition changes or device calibrations (Shifting)
      - Temporary AP outages or packet loss (RandomValueDropout)
    """
    def __init__(self, cfg: any):
        """
        Initializes the data augmentation pipeline using settings from a configuration object.
        
        Args:
            cfg (any): A configuration object (e.g., box.Box) containing nested 
                       parameters for `noise_level`, `max_shift`, and `probability`
                       for each augmentation type.
        """
        # Compose groups the augmentations together so they are applied sequentially.
        # It reads probabilities and constraints from the global configuration.
        self.transform = transforms.Compose([
            # 1. Add Gaussian noise to simulate minute measurement fluctuations
            GaussianNoise(
                std_range=(0.0, cfg.data_augmentation.noise.noise_level), 
                p=cfg.data_augmentation.noise.probability
            ),
            
            # 2. Shift the overall signal up or down to simulate baseline transmission offset
            Shifting(
                shift=cfg.data_augmentation.shift.max_shift, 
                p=cfg.data_augmentation.shift.probability
            ),
            
            # 3. Drop random RSSI values to simulate failing / temporarily blocked Access Points
            RandomValueDropout(
                p=cfg.data_augmentation.dropout.probability
            )
        ])

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        """
        Applies the augmentation pipeline to the input RSSI tensor.
        
        Args:
            x (torch.Tensor): The original normalized RSSI signal tensor.
                              Expected shape depends on the batch size and AP count.
        
        Returns:
            torch.Tensor: The augmented tensor, with modifications applied probabilistically 
                          based on the configuration.
        """
        # Pass the input tensor through the composed sequence of transformations
        return self.transform(x)

### Dataset

In [34]:
class ProjectDataset(Dataset):
    """
    A PyTorch Dataset class designed specifically for Wi-Fi RSSI indoor localization.
    
    This dataset handles loading raw CSV files (like UJIndoorLoc), preprocessing 
    the RSSI signal strengths, normalizing geographic coordinates (Latitude/Longitude), 
    and yielding samples as PyTorch tensors ready for model training.
    """
    def __init__(self, cfg: Any,
                 split: str,
                 sample_transform: transforms = None,
                 label_transform: transforms = None
        ) -> None:
        """
        Initializes the localization dataset.

        Args:
            cfg (Any): A configuration object containing dataset parameters like file paths,
                       random seeds, geographical bounds, and 'no-signal' values.
            split (str): Indicates which data subset to load. Must be 'training', 'validation', or 'test'.
            sample_transform (transforms, optional): PyTorch transform(s) applied to the RSSI input matrices 
                                                     (such as the `DataAugmentation` pipeline).
            label_transform (transforms, optional): PyTorch transform(s) applied to the target labels.
        """
        self.split = split
        self.random_seed = cfg.seed
        self.cfg = cfg.dataset
        self.data_root = Path(self.cfg.root_path)
        self.sample_transform = sample_transform
        self.label_transform = label_transform

        # Construct absolute paths for training and testing data files
        self.training_data_path = self.data_root / self.cfg.training_data_name
        self.test_data_path = self.data_root / self.cfg.validation_data_name

        # Ensure dataset files are physically present before loading
        if not self.training_data_path.exists():
            raise FileNotFoundError(f"Training data file not found at {self.training_data_path}")
        if not self.test_data_path.exists():
            raise FileNotFoundError(f"Test data file not found at {self.test_data_path}")

        # _load_data orchestrates reading from disk, cleaning, and normalizing 
        self.samples, self.labels = self._load_data()

        # Sanity check to ensure we didn't end up with an empty subset
        if len(self.samples) == 0:
            raise RuntimeError(
                f"Dataset split '{split}' is empty. Check your CSV parsing or train/val split."
            )


    def _load_data(self) -> Tuple[np.ndarray, np.ndarray]:
        """
        Reads raw dataset files, maps missing signal markers, and normalizes inputs and labels.

        Returns:
            Tuple[np.ndarray, np.ndarray]:
                - samples: A 2D array of normalized RSSI values (shape: [num_samples, num_APs]).
                - labels: A 2D array containing normalized [Floor, Longitude, Latitude] targets.
        """
        # We drop fields that are irrelevant for RSSI-based neural network regression/classification
        drop_columns = ["RELATIVEPOSITION", "USERID", "PHONEID", "TIMESTAMP"]

        # 1. READ CSV DATA
        if self.split == 'test':
            test_data = pd.read_csv(self.test_data_path)
            data = test_data.drop(columns=drop_columns)
        elif self.split in ['training', 'validation']:
            training_data = pd.read_csv(self.training_data_path)
            data = training_data.drop(columns=drop_columns)
        else:
            raise ValueError(f"Invalid split '{self.split}'. Must be one of 'training', 'validation', or 'test'.")

        # 2. SIGNAL STRENGTH NORMALIZATION
        # Usually, +100 signifies 'No AP Detected' in this type of dataset.
        # We replace this with a physically meaningful default weak signal floor (e.g., -110 dBm)
        data = data.replace(100, self.cfg.no_signal_value)

        # Separate feature columns (APs) from label columns to apply Feature Scaling to RSSI values
        columns_to_normalize = data.columns.difference(['FLOOR', 'BUILDINGID', 'LATITUDE', 'LONGITUDE', 'SPACEID'])
        max_value = 0  # Max realistic Wi-Fi signal is 0 dBm
        min_value = self.cfg.no_signal_value
        
        # Min-Max Scale the RSSI values so they fall within [0.0, 1.0]
        data[columns_to_normalize] = (data[columns_to_normalize] - min_value) / (max_value - min_value)

        # 3. LABEL PREPARATION
        # Create a unique CLASSID mapping a specific location (Building + Space).
        # This is useful for stratified splitting and discrete location classification
        data['CLASSID'] = data['BUILDINGID'].astype(str) + '_' + data['SPACEID'].astype(str)
        data = data.drop(columns=['BUILDINGID', 'SPACEID'])

        # 4. COORDINATE NORMALIZATION
        # Ensuring no geographical outliers exist comparing against our config definitions
        if (data['LONGITUDE'] < self.cfg.longitude_min).any() or (data['LONGITUDE'] > self.cfg.longitude_max).any():
            raise ValueError("Longitude values are outside of the expected range.")
        if (data['LATITUDE'] < self.cfg.latitude_min).any() or (data['LATITUDE'] > self.cfg.latitude_max).any():
            raise ValueError("Latitude values are outside of the expected range.")

        # Linearly scale geographic metrics into the [0.0, 1.0] bounds for better neural network convergence
        data['LATITUDE'] = (data['LATITUDE'] - self.cfg.latitude_min) / (self.cfg.latitude_max - self.cfg.latitude_min)
        data['LONGITUDE'] = (data['LONGITUDE'] - self.cfg.longitude_min) / (self.cfg.longitude_max - self.cfg.longitude_min)

        # 5. SPLIT RESOLUTION
        if self.split in ['training', 'validation']:
            # Automatically carve out 10% of the training pool for validation, stratifying on CLASSID
            # so every specific building/space retains proportional representation
            training_data, validation_data = train_test_split(
                data,
                test_size=0.10,           
                random_state=self.random_seed,          
                stratify=data['CLASSID']  
            )

            # Assign the local 'data' variable depending on which split was requested
            if self.split == 'training':
                data = training_data
            else:
                data = validation_data

        # Finalize subsets
        label_columns = ["FLOOR", "LONGITUDE", "LATITUDE", "CLASSID"]
        labels = data.loc[:, label_columns].copy()
        samples = data.drop(columns=label_columns)

        return np.array(samples, dtype=np.float32), np.array(labels, dtype=np.float32)

    def __len__(self) -> int:
        """
        Returns the total number of samples present in this dataset subset.
        """
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Union[int, torch.Tensor]]:
        """
        Retrieves a single data point and its target labels at a given index.
        Applies any runtime data augmentations before yielding.

        Args:
            idx (int): Global index into the dataset array.

        Returns:
            Tuple containing:
                - x (torch.Tensor): Feature tensor representing Wi-Fi RSSI signals.
                - y (torch.Tensor/int): Target label tensor representing [Floor, Longitude, Latitude, ClassID]
        """
        # Convert raw numpy arrays to Torch tensors on-the-fly
        x = torch.from_numpy(self.samples[idx])
        y = torch.from_numpy(self.labels[idx])

        # Apply composed augmentations (like random dropouts or gaussian noise)
        if self.sample_transform:
            x = self.sample_transform(x)
            
        # Apply any targeted manipulations to labels (if provided)
        if self.label_transform:
            y = self.label_transform(y)

        return x, y

## Class for general utilities

In [35]:
class Utilities:
    """
    A unified helper class to encapsulate scattered training, logging, and evaluation utilities.
    
    This includes dataset loaders, reproducible random seeding, dynamically patching nested 
    configurations, calculating domain-specific location errors (meters), and analyzing 
    neural network parameter weights.
    """

    def __init__(self):
        """
        Initializes the Utilities class.
        """
        pass

    def seed_worker(self, worker_id):
        """
        Sets a specific worker seed for PyTorch DataLoaders when using multiprocessing (`num_workers > 0`).
        This guarantees that even with multiple data fetchers, data augmentation randomness 
        remains deterministic and strictly reproducible based on the central clock.
        
        Args:
            worker_id (int): ID of the worker thread. Required by PyTorch signature.
        """
        _ = worker_id  # Unused, but required by the DataLoader signature
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    def set_by_path(self, nested_box, path, value):
        """
        Dynamically modifies a deeply nested key inside a Box/dictionary using a dot-separated string.
        Useful when receiving unrolled hyperparameter updates from Ax (e.g. 'training.lr_scheduler.lr').
        
        Args:
            nested_box (box.Box): The configuration object to mutate.
            path (str): Dot-separated path to the parameter (e.g., 'data_augmentation.dropout.probability').
            value (any): The new value to set.
        """
        keys = path.split('.')
        # Uses functools.reduce to recursively drill down into the dictionary until the 
        # second-to-last key, then sets the final key directly.
        functools.reduce(operator.getitem, keys[:-1], nested_box)[keys[-1]] = value

    def get_dataset(self, cfg, split='training'):
        """
        Instantiates datasets and PyTorch DataLoaders based on the specified phase.
        Automatically attaches Data Augmentations *only* if the phase is 'training'.

        Args:
            cfg (Box): Active configuration object containing dataset sizes, seeds, and augmentations.
            split (str): 'training', 'validation', or 'test'.

        Returns:
            Tuple[ProjectDataset, DataLoader]: The initialized PyTorch Dataset and the batched DataLoader.
        """
        # Inject our augmentation pipeline only during the training phase
        if split == 'training':
            transform_ops = [
                DataAugmentation(cfg.dataset)
            ]
        else:  # No augmentations should be applied to validation or test data!
            transform_ops = []

        datapoint_transform = transforms.Compose(transform_ops)
        label_transform = transforms.Compose([])

        dataset = ProjectDataset(cfg, split, sample_transform=datapoint_transform,
                        label_transform=label_transform)

        # PyTorch Generator specifically assigned to the dataloader for internal shuffle determinism
        g = torch.Generator()
        g.manual_seed(cfg.seed)

        dataloader = DataLoader(
            dataset,
            batch_size=cfg.dataset.batch_size,
            shuffle=(split == 'training'), # Only randomize sequence order during optimization/training
            worker_init_fn=self.seed_worker,
            generator=g
        )

        return dataset, dataloader

    def set_seed(self, seed: int):
        """
        Locks all random mathematical operations globally directly to a specific seed.
        Ensures hardware (CUDA), math cores (Numpy), and core library (PyTorch) functions 
        will output the exact same results every run.

        Args:
            seed (int): The master random seed.
        """
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # Lock multi-GPU setups if present
        
        # Eliminate backend convolution variations (sacrifices small speed chunks for absolute determinism)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    def get_parameters_table(self, model: torch.nn.Module, position: int = 0) -> str:
        '''
        Args:
            model (torch.nn.Module): The PyTorch model to summarize
            position (int): The position in the model hierarchy (recursive depth)

        Returns a list of tuples containing (module name, position, number of trainable parameters).
        '''
        # name, position, number trainable parameters
        lines = []
        num_trainable_params = 0
        
        # Calculate parameters belonging inherently to this individual block
        for _, param in model.named_parameters(recurse=False):
            if param.requires_grad:
                num_trainable_params += math.prod(param.size())

        child_lines = []
        # Drill down into nested layers recursively
        for module in model.children():
            child_lines += self.get_parameters_table(module, position + 1)

        # Bubble up lower-level params to represent aggregate block sizes
        sum_params = 0
        for line in child_lines:
            if line[1] == position + 1:
                sum_params += line[2]

        if num_trainable_params == 0:
            lines.append([type(model).__name__, position, sum_params])
        else:
            lines.append([type(model).__name__, position, num_trainable_params])
            
        lines += child_lines

        return lines

    def get_model_summary(self, model: torch.nn.Module) -> str:
        """
        Combines the tree array generated by `get_parameters_table` into a human-readable 
        ASCII diagram.

        Args:
            model (torch.nn.Module): PyTorch neural net object.

        Returns:
            str: Pretty-printed hierarchy string.
        """
        lines = self.get_parameters_table(model)
        summary = ""
        for line in lines:
            # Indent each structural depth with spacing & block symbols
            summary += "  " * line[1] + f"└─ {line[0]}: {line[2]} trainable parameters\n"
        summary += "=================================================================\n"
        summary += f"Total Trainable Parameters: {lines[0][2]}\n"
        return summary
    
    def calc_meter_from_mae(self, cfg: Any, mae_lat: float, mae_lon: float) -> float:
        """
        Converts Neural Network normalized output Mean Absolute Error [0, 1] back into 
        physical distances (meters) on the UJIndoorLoc coordinate map (UTM-like Cartesian map).
        Calculates horizontal/vertical error directly (Manhattan Distance metric equivalent).

        Args:
            cfg (Any): Reference block for lat/lon boundaries.
            mae_lat (float): Normalized evaluation latitude prediction error.
            mae_lon (float): Normalized evaluation longitude prediction error.

        Returns:
            float: De-normalized error summation representing total horizontal offset in mapping grid.
        """
        # Fetch physical bounding boxes spanning the dataset map configuration
        lat_min, lat_max = cfg.dataset.latitude_min, cfg.dataset.latitude_max
        lon_min, lon_max = cfg.dataset.longitude_min, cfg.dataset.longitude_max

        # Scaling normalized distances mapping directly back to world dimensions
        mae_lat_meters = mae_lat * (lat_max - lat_min)
        mae_lon_meters = mae_lon * (lon_max - lon_min)

        total_mae_meters = mae_lat_meters + mae_lon_meters

        return total_mae_meters
    
    def calc_mpe_meters(self, cfg: Any, pred_coordinates: torch.Tensor, target_coordinates: torch.Tensor) -> float:
        '''
        Convert Mean Percentage Error (MPE) to a custom meter value for better interpretability.

        Args:
            cfg (Any): Configuration object containing the necessary parameters for conversion.
            pred_coordinates (torch.Tensor): Predicted coordinates.
            target_coordinates (torch.Tensor): Target coordinates.

        Returns:
            float: The sum of the mpe for this batch
            int: The number of samples in this batch
        '''
        lat_min, lat_max = cfg.dataset.latitude_min, cfg.dataset.latitude_max
        lon_min, lon_max = cfg.dataset.longitude_min, cfg.dataset.longitude_max

        # Upcast precision before denorm scalar multiplications
        pred_coordinates = pred_coordinates.cpu().to(torch.float32)
        target_coordinates = target_coordinates.cpu().to(torch.float32)

        # Denormalize longitude and latitude arrays respectively
        pred_lat_m = pred_coordinates[:, 1] * (lat_max - lat_min) + lat_min
        pred_lon_m = pred_coordinates[:, 0] * (lon_max - lon_min) + lon_min

        target_lat_m = target_coordinates[:, 1] * (lat_max - lat_min) + lat_min
        target_lon_m = target_coordinates[:, 0] * (lon_max - lon_min) + lon_min
        
        # Merge individual dimensions back into a 2D Tensor Grid for hypotenuse measuring
        preds_meters = torch.stack([pred_lat_m, pred_lon_m], dim=1)
        targets_meters = torch.stack([target_lat_m, target_lon_m], dim=1)
        
        # Calculate torchmetrics pairwise distance for this batch
        dist_matrix = pairwise_euclidean_distance(preds_meters, targets_meters, reduction="none")
        
        # Diagonal contains euclidean errors per sample in current batch
        batch_distances = torch.linalg.vector_norm(preds_meters - targets_meters, ord=2, dim=1)
        
        # Sum errors and count samples
        current_batch_size = target_coordinates.size(0)
        batch_mpe_sum = torch.sum(batch_distances).item()

        return batch_mpe_sum, current_batch_size

    def save_best_parameters(self, best_params: Box, filename: str = "best_parameters.yaml"):
        '''
        Save the best parameters to a YAML file.

        Args:
            best_params (Dict[str, Any]): Dictionary containing the best parameters to save.
            filename (str): The name of the YAML file to save the parameters to.
        '''

        with open(filename, "w") as f:
            yaml.dump(best_params.to_dict(), f)

## Evaluator class
Class for handling the evaluation of the model with the test set, calculating different relevant metrics

In [36]:
class Evaluator:
    """
    Evaluator for binary and multiclass classification tasks.

    Handles computing both classification performance (predicting Building/Floor) 
    and regression precision (predicting local X/Y coordinates).
    It manages the instantiations of metric tracking objects, accumulates metrics 
    batch-by-batch safely via `torchmetrics`, and logs the final validation statistics 
    back to Weights & Biases (WandB).
    """

    def __init__(
        self,
        cfg: Any,
        eval_loader: DataLoader,
        model: nn.Module,
        device: torch.device,
        wb_run: Any

    ):
        """
        Initializes the evaluation environment.

        Args:
            cfg (Any): Configuration object (containing `floor_classes` and loss weights).
            eval_loader (DataLoader): The validation or test split PyTorch DataLoader.
            model (nn.Module): The active network architecture to evaluate.
            device (torch.device): Compute hardware ('cuda' or 'cpu').
            wb_run (Any): Active Weights & Biases session instance for logging validation metrics.
        """
        self.cfg = cfg
        self.eval_loader = eval_loader
        self.model = model
        self.device = device
        self.wb_run = wb_run

        # Assign loss formulas based on the architectural layout of the requested model
        if self.model.get_name() == "Ensemble_MLP":
            # Traditional Dual-Head network mapping features directly to labels and coordinates
            self.criterion_classification = torch.nn.CrossEntropyLoss()
            self.criterion_regression = torch.nn.L1Loss() # Mean Absolute Error for coordinate loss
            self.classification_loss_weight = cfg.model.loss_weights.classification
            self.regression_loss_weight = cfg.model.loss_weights.regression
            
        elif self.model.get_name() == "Autoencoder":
            # Tri-Head network; attempts to classify, regress, AND reconstruct the original RSSI signal
            self.criterion_classification = torch.nn.NLLLoss()
            self.criterion_regression = torch.nn.MSELoss()
            self.criterion_decoder = torch.nn.MSELoss() # Reconstruction loss
            self.classification_loss_weight = cfg.model.loss_weights.classification
            self.regression_loss_weight = cfg.model.loss_weights.regression
            self.decoder_loss_weight = cfg.model.loss_weights.decoder

        # Verify configuration validity
        if not hasattr(cfg, 'floor_classes'):
            raise ValueError(
                "Config object 'cfg' must contain 'floor_classes' attribute.")
        self.num_classes = cfg.floor_classes

        # Setup standard dictionary signature for all PyTorch Multiclass metrics
        self.task_kwargs = {"num_classes": self.num_classes}

        # Initialize TorchMetrics inside a ModuleDict so they auto-transfer to CUDA when called
        self.metrics = nn.ModuleDict()
        self.metrics.update(self._get_metrics())
        for metric in self.metrics.values():
            metric.to(device)

    def _get_metrics(self) -> dict:
        """
        Build the dictionary of evaluation metrics.

        Regression metrics:
            - MSE
            - MAE

        Multiclass metrics:
            - Accuracy (micro-averaged)
            - F1, AUROC (macro-averaged)
            - Confusion Matrix

        Returns:
            dict: Dictionary mapping metric names to torchmetrics instances.
        """
        # Metrics for multiclass classification and regression
        macro_kwargs = {**self.task_kwargs, "average": "macro"}
        micro_kwargs = {**self.task_kwargs, "average": "micro"}
        
        metrics = {
            'conf_matrix': MulticlassConfusionMatrix(**self.task_kwargs),
            'accuracy_evaluation': MulticlassAccuracy(**micro_kwargs),
            'auroc_macro': MulticlassAUROC(**macro_kwargs),
            'f1_macro': MulticlassF1Score(**macro_kwargs),
            'mse': MeanSquaredError(),
            'mae': MeanAbsoluteError(),
            'mae_lat': MeanAbsoluteError(), # Independent tracking for latitude
            'mae_lon': MeanAbsoluteError()  # Independent tracking for longitude
        }

        return metrics

    def _reset_metrics(self) -> None:
        """
        Clears accumulated statistical states from metric objects at the start of an evaluation epoch.
        """
        for metric in self.metrics.values():
            metric.reset()

    @torch.no_grad()
    def eval(self, return_metrics: bool = False) -> None:
        """
        Executes a complete forward-pass sweep across the validation/test dataset.

        Args:
            return_metrics (bool): Determine whether the dictionary of results should be returned 
                                   (useful for Hyperparameter Optimization loops). Defaults to False.

        Returns dict if `return_metrics` is True.
        """
        # Lock model architectures to disable Dropouts and batch-normalization variations
        self.model.eval()
        self._reset_metrics()

        running_loss = 0.0
        running_mpe_sum = 0.0
        total_samples = 0

        util = Utilities()

        # Step through evaluation batches one by one without tracking gradients
        for _, (x, y_true) in enumerate(self.eval_loader):
            x, y_true = x.to(self.device), y_true.to(self.device)
            
            # Unpack targets (Index 0 is Floor/Class, Indices 1/2 are X/Y Coordinates)
            y_floor = y_true[:, 0].long()
            y_coordinates = y_true[:, 1:3]

            # --- Forward Pass and Loss Accumulation ---
            if self.model.get_name() == "Ensemble_MLP":
                y_pred_classification, y_pred_regression = self.model(x)

                y_pred_classification_probs = torch.softmax(y_pred_classification, dim=1)
                y_pred_classification_labels = torch.argmax(y_pred_classification, dim=1)

                loss_classification = self.criterion_classification(y_pred_classification, y_floor)
                loss_regression = self.criterion_regression(y_pred_regression, y_coordinates)

                # Weight-balance dual loss based on configuration parameters
                loss = self.classification_loss_weight * loss_classification + self.regression_loss_weight * loss_regression
                
            elif self.model.get_name() == "Autoencoder":
                y_pred_classification, y_pred_regression, y_pred_decoder = self.model(x)

                y_pred_classification_probs = torch.softmax(y_pred_classification, dim=1)
                y_pred_classification_labels = torch.argmax(y_pred_classification, dim=1)

                loss_classification = self.criterion_classification(y_pred_classification, y_floor)
                loss_regression = self.criterion_regression(y_pred_regression, y_coordinates)
                loss_decoder = self.criterion_decoder(y_pred_decoder, x) # Compare decoded output against original input

                # Tri-Weighted loss
                loss = self.classification_loss_weight * loss_classification + self.regression_loss_weight * loss_regression + self.decoder_loss_weight * loss_decoder
            else:
                raise NotImplementedError(f"Model {self.model.get_name()} not implemented in training loop.")

            running_loss += loss.item()

            # --- Update Batch Metrics ---
            for metric_name, metric in self.metrics.items():
                if isinstance(metric, MulticlassAUROC):
                    # AUROC requires probabilistic probabilities as inputs, not discrete labels
                    metric.update(y_pred_classification_probs, y_floor)
                elif isinstance(metric, MulticlassConfusionMatrix) or isinstance(metric, MulticlassAccuracy) or isinstance(metric, MulticlassF1Score):
                    metric.update(y_pred_classification_labels, y_floor)
                # Regression metrics
                elif isinstance(metric, MeanSquaredError) or isinstance(metric, MeanAbsoluteError):
                    # Route Coordinate subset arrays to the specialized longitude/latitude tracking objects
                    if metric_name == 'mae_lon':
                        metric.update(y_pred_regression[:, 0].contiguous(), y_coordinates[:, 0].contiguous())
                    elif metric_name == 'mae_lat':
                        metric.update(y_pred_regression[:, 1].contiguous(), y_coordinates[:, 1].contiguous())
                    else:
                        metric.update(y_pred_regression, y_coordinates.contiguous())
                else:
                    logging.error(f"Metric {metric_name} not implemented in evaluation loop routing.")
            
            # Leverage Utilities to calculate physical distance errors metrics in terms of meters.
            batch_mpe_sum, batch_size = util.calc_mpe_meters(self.cfg, y_pred_regression, y_coordinates)
            running_mpe_sum += batch_mpe_sum
            total_samples += batch_size

        # Compute final metrics for hole dataset after iterating through all batches
        final_metrics = {}

        # Calculate average loss per batch
        final_metrics = {
            'total_loss_evaluation': running_loss / len(self.eval_loader)}

        # Tell torchmetrics to collapse all accumulated batch steps into single dataset-level measurements
        for metric_name, metric in self.metrics.items():
            try:
                final_metrics[metric_name] = metric.compute()
            except Exception as e:
                logging.warning("[Warning] Could not compute metric '%s'. Set to None. Error: %s",
                                metric_name, e)
                final_metrics[metric_name] = None
                
        # Only print telemetry physically to terminal if we aren't drowning the terminal in AX iterations
        if not self.cfg.optimization.active:
            print(f"Evaluation accuracy: {final_metrics['accuracy_evaluation'].item()}")
        
        # Format custom physical MPE (Euclidean hypotenuse error) for the dataset
        final_mpe = running_mpe_sum / total_samples
        final_metrics['mpe_meters'] = final_mpe

        # Calculate the custom meter value from the MAE of latitude and longitude
        mae_lat = final_metrics['mae_lat'].item()
        mae_lon = final_metrics['mae_lon'].item()
        
        final_metrics['mae_meters'] = util.calc_meter_from_mae(self.cfg, mae_lat, mae_lon)
        
        if not self.cfg.optimization.active:
            print(f"Evaluation MPE in meters: {final_metrics['mpe_meters']}")
            print(f"Evaluation MAE in meters: {final_metrics['mae_meters']}")

        # Remove mae_lat and mae_lon from final_metrics
        del final_metrics['mae_lat']
        del final_metrics['mae_lon']

        if self.wb_run is not None:
            self.wb_run.log(final_metrics)

        if return_metrics:
            return final_metrics

## Trainer class
Class for handling the training of the model

In [37]:
class Trainer:
    """
    Core orchestrator class for executing the backpropagation training loop.
    
    Responsibilities include:
      - Initializing parameter groups, learning rates, and optimizers (AdamW default).
      - Managing learning rate scheduling strategies (e.g. Cosine Annealing).
      - Executing batch-wise forward passes, gradient calculations, and loss step updates.
      - Freezing portions of the neural network structurally throughout training 
        (like freezing the Autoencoder features dynamically near the end of training).
      - Interfacing directly with `Evaluator` at checkpoint intervals.
    """

    def __init__(
        self,
        cfg: any, 
        train_loader: DataLoader,
        model: nn.Module,
        evaluator: Evaluator,
        device: torch.device,
        wb_run: Any
    ):
        """
        Setup tracking instances and hyperparameter defaults for the Trainer.

        Args:
            cfg (object): Layered Box config holding learning rates, weight decay, epochs, and frozen states.
            train_loader (DataLoader): PyTorch iterator generating training batches.
            model (nn.Module): Instantiated algorithm being trained.
            evaluator (Evaluator): Handles periodic validation sweeps.
            device (torch.device): 'cuda' or 'cpu'.
            wb_run (Any): Active Weights and Biases interface for charting loss.
        """
        self.train_loader = train_loader

        # Organize deep-nested config variables cleanly into a single TrainingParameters structure 
        self.training_parameters = TrainingParameters(
            lr=cfg.training.learning_rate,
            lr_min=cfg.training.learning_rate_min,
            lr_scheduler=cfg.training.learning_rate_scheduler,
            beta1=cfg.training.beta1,
            beta2=cfg.training.beta2,
            weight_decay=cfg.training.weight_decay,
            epochs=cfg.training.num_epochs,
        )

        # We deploy AdamW (Adam with Weight Decay coupling) standard for modern tabular optimization
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=self.training_parameters.get_learning_rate(),
            betas=self.training_parameters.get_betas(),
            weight_decay=self.training_parameters.get_weight_decay()
        )

        # Component Initialization
        model.to(device)
        self.model = model
        self.optimizer = optimizer
        self.evaluator = evaluator
        self.device = device

        # Fetch logging frequency boundaries
        self.log_interval = cfg.logging.log_interval
        self.eval_interval = cfg.logging.eval_interval

        # Architecture-aware loss injection (mirroring logic inside the Evaluator module)
        if self.model.get_name() == "Ensemble_MLP":
            self.criterion_classification = torch.nn.CrossEntropyLoss()
            self.criterion_regression = torch.nn.L1Loss()
            self.classification_loss_weight = cfg.model.loss_weights.classification
            self.regression_loss_weight = cfg.model.loss_weights.regression
            
        elif self.model.get_name() == "Autoencoder":
            self.criterion_classification = torch.nn.NLLLoss()
            self.criterion_regression = torch.nn.MSELoss()
            self.criterion_decoder = torch.nn.MSELoss()
            
            self.classification_loss_weight = cfg.model.loss_weights.classification
            self.regression_loss_weight = cfg.model.loss_weights.regression
            self.decoder_loss_weight = cfg.model.loss_weights.decoder
            
            # Extracts probability (0.0 to 1.0) marking WHEN the Autoencoder block should freeze
            # e.g., if freeze_ae_training = 0.2, the AE freezes during the last 20% of epochs 
            # to let the classification/regression heads fine-tune on completely static embeddings.
            self.freeze_ae_training = cfg.model.autoencoder.freeze_ae_training

        # Setup learning rate decaying to smoothly lower LR as model approaches global minimum
        self.scheduler = None
        if self.training_parameters.lr_scheduler == "cosine":
            self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer,
                T_max=self.training_parameters.get_epochs(),
                eta_min=self.training_parameters.get_learning_rate_min()
            )

        self.wb_run = wb_run
        self.cfg = cfg


    def get_trained_model(self) -> nn.Module:
        """
        Retrieves the locally trained neural network model layout (usually after a full train loop completes).
        """
        return self.model


    def train(self):
        """
        Primary execution loop for training. Overarches epochs, dataset loader iterating, gradient descents,
        and scheduler adjustments. Periodically jumps to external class references like the Evaluator.
        """
        for epoch in range(self.training_parameters.get_epochs()):
            start_time = time.time()
            
            # Enable features like PyTorch dropout/batchnorm which modify forward passes structurally
            self.model.train()
            running_loss = 0
            correct = 0
            total = 0

            # Step batch-wise through available training data
            for step, (x, y_true) in enumerate(self.train_loader):
                x, y_true = x.to(self.device), y_true.to(self.device)
                
                # Split Targets: Building/Floor identifier vs 2D Plane Coordinate
                y_floor = y_true[:, 0].long()
                y_coordinates = y_true[:, 1:3]

                # Purge prior step gradients 
                self.optimizer.zero_grad()

                # --- 1) Dual Head Architecture Training Step ---
                if self.model.get_name() == "Ensemble_MLP":
                    y_pred_classification, y_pred_regression = self.model(x)
                    
                    loss_classification = self.criterion_classification(y_pred_classification, y_floor)
                    loss_regression = self.criterion_regression(y_pred_regression, y_coordinates)

                    loss = self.classification_loss_weight * loss_classification + self.regression_loss_weight * loss_regression
                
                # --- 2) Tri-Head Autoencoder Embedding Training Step ---
                elif self.model.get_name() == "Autoencoder":
                    y_pred_classification, y_pred_regression, y_pred_decoder = self.model(x)

                    loss_classification = self.criterion_classification(y_pred_classification, y_floor)
                    loss_regression = self.criterion_regression(y_pred_regression, y_coordinates)
                    
                    # Logic block checking if we have bypassed the structural freeze threshold
                    if epoch > self.training_parameters.get_epochs() * (1 - self.freeze_ae_training):
                        # Force PyTorch to disable graphing node calculation on decoder output
                        with torch.no_grad():
                            loss_decoder = self.criterion_decoder(y_pred_decoder, x)
                            
                        # Fully lock parameter gradient updates mapping the feature embedding architecture
                        for param in self.model.encoder.parameters():
                            param.requires_grad = False
                        for param in self.model.decoder.parameters():
                            param.requires_grad = False
                    else:
                        # Otherwise, calculate decoder loss against pure data inputs normally
                        loss_decoder = self.criterion_decoder(y_pred_decoder, x)

                    loss = self.classification_loss_weight * loss_classification + self.regression_loss_weight * loss_regression + self.decoder_loss_weight * loss_decoder

                else:
                    raise NotImplementedError(f"Model {self.model.get_name()} not implemented in training loop.")

                # Backward pass accumulating partial graph derivative logic into .grad tensors
                loss.backward()

                # Cap gradient explosives explicitly preventing NaN instability spikes early in training
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

                # Finalize matrix updates
                self.optimizer.step()

                running_loss += loss.item() * x.size(0)

                # Calculate base discrete metric to gauge training progress instantly (micro-accuracy)
                predicted_classes = torch.argmax(y_pred_classification, dim=1)
                total += y_floor.size(0)
                correct += (predicted_classes == y_floor).sum().item()

                # Telemetry Upload 
                if self.wb_run is not None and ((step + 1) % self.log_interval == 0):
                    accuracy = correct / total if total > 0 else 0
                    current_lr = self.optimizer.param_groups[0]['lr']

                    self.wb_run.log({
                        "floor_accuracy_training": accuracy,
                        "coordinates_mse_training": loss_regression.item() * self.regression_loss_weight,
                        "total_loss_training": loss.item(),
                        "classification_loss_training": loss_classification.item() * self.classification_loss_weight,
                        "learning_rate": current_lr
                    })

            end_time = time.time()
            if self.wb_run is not None:
                self.wb_run.log({"epoch_duration": end_time - start_time})

            # Alter learning curve drop based on scheduler state variables
            if self.scheduler is not None:
                self.scheduler.step()

            # Prevent terminal command line spamming purely if we're in Ax Hyperoptimization mapping 
            if not self.cfg.optimization.active:
                print("Epoch [{0}/{1}]".format(epoch+1, self.training_parameters.get_epochs()))

            # Execute validation sweep
            if (epoch + 1) % self.eval_interval == 0:
                self.model.eval()
                with torch.no_grad():
                    self.evaluator.eval()

        logging.info("Training finished")
        
        # Determine if we should push final validation dict outward (Required by Ax Bayesian Ops)
        return_metrics = self.cfg.optimization.active
        if return_metrics:
            return self.evaluator.eval(return_metrics)

## Model
Class of the selected model including the definition of the architecture as well as the forward porapagation

### Simple MLP
Simple MLP model for comparison baseline and to test code early during development.

In [38]:
class Classifier(nn.Module):
    '''
    MLP to classify the Floor..
    '''
    def __init__(self, cfg: Any):
        '''
        Initializes the model based on the configuration.
        '''
        super().__init__()
        self.name = "Classifier"
        self.layers = nn.Sequential(
            nn.Linear(cfg.input_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(128, cfg.floor_classes)
        )

    def get_name(self) -> str:
        '''
        Returns the name of the model.
        '''
        return self.name

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Forward pass of the model.

        Args:
            x: Input tensor

        Returns:
            Output tensor after passing through the model.
        '''
        x = self.layers(x)
        # Softmax for classification output
        x = torch.softmax(x, dim=1)
        return x

class Regressor(nn.Module):
    '''
    MLP to regress the longitude and latitude.
    '''
    def __init__(self, cfg: Any):
        '''
        Initializes the model based on the configuration.
        '''
        super().__init__()
        self.name = "Regressor"
        self.layers = nn.Sequential(
            nn.Linear(cfg.input_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(128, 2)
        )

    def get_name(self) -> str:
        '''
        Returns the name of the model.
        '''
        return self.name

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Forward pass of the model.

        Args:
            x: Input tensor

        Returns:
            Output tensor after passing through the model.
        '''
        x = self.layers(x)
        # Ensure output is between 0 and 1 for normalized coordinates
        x = torch.sigmoid(x)
        return x

class Ensemble_MLP(nn.Module):
    '''
    Exemple for a simple MLP model consisting of a classification for the FLoor and a regression for the longitude and latitude.
    '''
    def __init__(self, cfg: Any):
        '''
        Initializes the model based on the configuration.
        '''
        super().__init__()
        self.name = "Ensemble_MLP"

        self.classifier = Classifier(cfg.classifier)
        self.regressor = Regressor(cfg.regressor)

    def get_name(self) -> str:
        '''
        Returns the name of the model.
        '''
        return self.name

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Forward pass of the hybrid model.

        Args:
            x: Input tensor

        Returns:
            Output tensor after passing through the model.
        '''

        y_pred_classification = self.classifier(x)
        y_pred_regression = self.regressor(x)

        return y_pred_classification, y_pred_regression

### Autoencoder
Autoencoder model based on paper "Indoor Localization With an Autoencoder-Based
Convolutional Neural Network" by Hatice Arslantas and Selcuk Okdem.

In [39]:
class AEClassifier(nn.Module):
    '''
    Classify the Floor..
    '''
    def __init__(self, cfg: Any):
        '''
        Initializes the model based on the configuration.
        '''
        super().__init__()
        self.name = "AEClassifier"
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=2),
            nn.BatchNorm2d(4),
            nn.LeakyReLU(),
            # Spatial dimension: 19

            nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=2),
            nn.BatchNorm2d(8),
            nn.LeakyReLU(),
            # Spatial dimension: 9

            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=2),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(),
            # Spatial dimension: 4

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=2),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            # Spatial dimension: 1

            nn.Flatten(),
            nn.Linear(1 * 1 * 32, cfg.classifier.floor_classes)
        )

    def get_name(self) -> str:
        '''
        Returns the name of the model.
        '''
        return self.name

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Forward pass of the model.

        Args:
            x: Input tensor

        Returns:
            Output tensor after passing through the model.
        '''
        x = self.layers(x)
        x = torch.log_softmax(x, dim=1)
        return x

class AERegressor(nn.Module):
    '''
    Regress the longitude and latitude.
    '''
    def __init__(self, cfg: Any):
        '''
        Initializes the model based on the configuration.
        '''
        super().__init__()
        self.name = "AERegressor"
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=2),
            nn.BatchNorm2d(4),
            nn.LeakyReLU(),
            # Spatial dimension: 19

            nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=2),
            nn.BatchNorm2d(8),
            nn.LeakyReLU(),
            # Spatial dimension: 9

            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=2),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(),
            # Spatial dimension: 4

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=2),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            # Spatial dimension: 1

            nn.Flatten(),
            nn.Linear(1 * 1 * 32, 2)
        )

    def get_name(self) -> str:
        '''
        Returns the name of the model.
        '''
        return self.name

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Forward pass of the model.

        Args:
            x: Input tensor

        Returns:
            Output tensor after passing through the model.
        '''
        x = self.layers(x)
        x = torch.sigmoid(x)
        return x

class Encoder(nn.Module):
    """
    Compresses flat 1D raw dataset input (Access Points) into a compact conceptual latent mapping.
    This bottleneck forces the network to learn only the most statistically meaningful correlations.
    """
    def __init__(self, cfg: Any):
        '''
        Initializes the model based on the configuration.
        '''
        super().__init__()
        self.name = "Encoder"
        
        # We enforce a perfect square latent dimension so we can easily map this output 
        # into a 2D image matrix for the subsequent CNN heads (e.g. 39 squared = 1521).
        self.latent_dim = pow(cfg.cnn_in_dim, 2)

        self.layers = nn.Sequential(
            nn.Linear(cfg.input_dim, cfg.autoencoder.fc_enc_dim),
            nn.BatchNorm1d(cfg.autoencoder.fc_enc_dim),
            nn.LeakyReLU(),

            nn.Linear(cfg.autoencoder.fc_enc_dim, self.latent_dim),
            nn.BatchNorm1d(self.latent_dim)
        )

    def get_name(self) -> str:
        '''
        Returns the name of the model.
        '''
        return self.name

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        '''
        Forward pass of the model.

        Args:
            input: Input tensor

        Returns:
            Output tensor after passing through the model.
        '''
        enc = self.layers(input)
        enc = torch.tanh(enc)
        return enc
    
class Decoder(nn.Module):
    """
    Attempts to structurally reconstruct the original 1D RSSI signal from the dense latent subspace.
    If the MSE loss of this module is low, it proves the Encoder mapping reflects reality accurately.
    """
    def __init__(self, cfg: Any):
        '''
        Initializes the model based on the configuration.
        '''
        super().__init__()
        self.name = "Decoder"
        self.latent_dim = pow(cfg.cnn_in_dim, 2)

        self.layers = nn.Sequential(
            nn.Linear(self.latent_dim, cfg.autoencoder.fc_dec_dim),
            nn.BatchNorm1d(cfg.autoencoder.fc_dec_dim),
            nn.LeakyReLU(),

            nn.Linear(cfg.autoencoder.fc_dec_dim, cfg.input_dim),
            nn.BatchNorm1d(cfg.input_dim)
        )

    def get_name(self) -> str:
        '''
        Returns the name of the model.
        '''
        return self.name

    def forward(self, enc: torch.Tensor) -> torch.Tensor:
        '''
        Forward pass of the model.

        Args:
            enc: Encoded input tensor

        Returns:
            Output tensor after passing through the model.
        '''
        dec = self.layers(enc)
        # Ensure output is between 0 and 1 for normalized coordinates
        dec = torch.sigmoid(dec)
        return dec

class AutoEncoder(nn.Module):
    def __init__(self, cfg: Any):
        super().__init__()
        self.name = "Autoencoder"

        self.classifier = AEClassifier(cfg)
        self.regressor = AERegressor(cfg)
        self.encoder = Encoder(cfg)
        self.decoder = Decoder(cfg)
        
        self.cnn_in_dim = cfg.cnn_in_dim

    def get_name(self) -> str:
        '''
        Returns the name of the model.
        '''
        return self.name

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Forward pass of the hybrid model.

        Args:
            x: Input tensor

        Returns:
            Output tensor after passing through the model.
        '''

        z = self.encoder(x)

        batch_size = int(z.numel() / pow(self.cnn_in_dim, 2))
        cnn_in_tensor = z.view(batch_size, 1, self.cnn_in_dim, self.cnn_in_dim)
        
        y_pred_classification = self.classifier(cnn_in_tensor)
        y_pred_regression = self.regressor(cnn_in_tensor)
              
        x_pred_ae = self.decoder(z)

        return y_pred_classification, y_pred_regression, x_pred_ae

## Main function

In [40]:
def main():
    """
    Main orchestration function to run a single, fully monitored training pipeline.
    
    This function coordinates the complete lifecycle of a standard PyTorch experiment:
      1. Hardware selection (CUDA/CPU/MPS) and random seeding.
      2. Instantiating Dataset and DataLoaders for train/validation/test splits.
      3. Constructing the dynamic neural network layout defined in the configuration.
      4. Logging hyperparameters to Weights & Biases (WandB) for cloud tracking.
      5. Executing the training loop via `Trainer` and validating via `Evaluator`.
      6. Verifying the final converged weights exclusively against the unseen `test` partition.
    """
    # Load global layered Box configuration 
    cfg = CONFIG
    if cfg is None:
        raise ValueError("Config dictionary must be provided.")

    # Create utility instance to handle seed locking and mapping math
    util = Utilities()

    # Lock PyTorch and Numpy hardware logic to ensure exact reproducible results
    util.set_seed(cfg.seed)

    # 1. Device Selection 
    # Picks the most optimal compute backend available on the user's specific machine
    device = torch.device('cpu')
    if torch.cuda.is_available():
        device = torch.device('cuda')       # NVIDIA GPUs
    elif torch.backends.mps.is_available():
        device = torch.device('mps')        # Apple Silicon (M1/M2/M3)
    logging.info('Using device: %s', device)

    # 2. Dataset Initialization 
    # Yields the PyTorch classes needed to step through the data in shuffled batches
    _, train_dataloader = util.get_dataset(cfg, split='training')
    _, eval_dataloader = util.get_dataset(cfg, split='validation')
    _, test_dataloader = util.get_dataset(cfg, split='test')

    # 3. Model Architecture Instantiation
    # Reads the config to determine which class definition should be utilized
    if cfg.model.name == "Ensemble_MLP":
        # Dual-head setup
        model = Ensemble_MLP(cfg.model)
    elif cfg.model.name == "Autoencoder":
        # Tri-head setup mapping the latent bottleneck
        model = AutoEncoder(cfg.model)
    else:
        raise NotImplementedError(f"Model {cfg.model.name} not implemented in training loop.")

    # Prints a human-readable topology of all trainable parameters
    model_summary = util.get_model_summary(model)
    print(model_summary)

    # 4. Weights and Biases (WandB) Telemetry 
    # Connects to the cloud to chart training metrics dynamically if toggled 'on'
    if cfg.logging.use_wandb:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M')
        wandb.login(key=cfg.logging.api_key)
        
        # Structure the run name neatly utilizing user flags and timestamps
        additional = cfg.logging.run_name_additional
        if additional != 'None' and additional is not None:
            run_name = f"{model.get_name()}_{additional}_{timestamp}"
        else:
            run_name = f"{model.get_name()}_{timestamp}"
            
        wb_run = wandb.init(
            # Cloud destination configurations
            entity=cfg.logging.team_name,
            project=cfg.logging.project_name,
            name=run_name,
            
            # Map configuration inputs visually to the cloud dashboard for future filtering
            config={
                "learning_rate": cfg.training.learning_rate,
                "learning_rate scheduler": cfg.training.learning_rate_scheduler,
                "architecture": model.get_name(),
                "dataset": "UjiIndoorLoc",
                "epochs": cfg.training.num_epochs,
                "model settings": {**dict(cfg.model)},
                "model architecture": model_summary
            }
        )
    else:
        wb_run = None

    # 5. Core execution logic mappings
    # Prepares validation step loops
    evaluator = Evaluator(cfg, eval_dataloader, model, device, wb_run)

    # Wraps the core PyTorch training loop
    trainer = Trainer(cfg,
                      train_dataloader,
                      model,
                      evaluator,
                      device,
                      wb_run)

    # Commences Epoch batch looping 
    trainer.train()

    # 6. Final Blind Assessment 
    # Evaluates the functionally completed network purely against the held-out "test" set
    test_evaluator = Evaluator(cfg, test_dataloader, trainer.get_trained_model(), device, wb_run)
    print("Running Evaluation")
    test_evaluator.eval()

    print("Finished")

## Bayesian optimizer
To optimize the hyperparameters of the model and achieve better performance

In [41]:
def optimize():
    """
    Executes a comprehensive Bayesian Hyperparameter Optimization framework utilizing the Meta Ax platform.
    
    This function iteratively searches a pre-defined multi-dimensional parameter space (e.g., learning 
    rates, dropout probabilities, shift variables) testing them functionally by continuously training 
    and resetting models, searching for the mathematically optimal network configuration.
    
    After finding the Pareto Frontier optimal hyperparameters, it natively launches a final 
    evaluative run benchmarking that peak model precisely against the unseen test dataset.
    """
    cfg = CONFIG
    if cfg is None:
        raise ValueError("Config dictionary must be provided.")

    # Instantiate helpers for parsing data hierarchies and deterministic mapping
    util = Utilities()
    util.set_seed(cfg.seed)

    # 1. Device Assignment (CUDA/MPS/CPU)
    device = torch.device('cpu')
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    logging.info('Using device: %s', device)

    # 2. Extract Data Split Iterators
    _, train_dataloader = util.get_dataset(cfg, split='training')
    _, eval_dataloader = util.get_dataset(cfg, split='validation')
    _, test_dataloader = util.get_dataset(cfg, split='test')

    # 3. Setup the Bayesian Experiment Environment
    ax_client = AxClient()

    # Deconstruct nested config constraints to form quantifiable minimization/threshold objectives 
    ax_objectives = {
        name: ObjectiveProperties(
            minimize=props["minimize"], 
            threshold=props.get("threshold")
        )
    for name, props in cfg.optimization.optimization_config.objectives.items()
    }

    # Bind the parameter boundaries (e.g. valid Dropout range [0.1, 0.5]) explicitly to the client
    ax_client.create_experiment(
        name="bayesian_optimization",
        parameters=cfg.optimization.search_space,
        objectives=ax_objectives,
        overwrite_existing_experiment=True
    )

    # 4. Core Optimization Execution Loop
    for i in range(cfg.optimization.optimization_config.total_trials):
        print(f"Starting Trial {i + 1}/{cfg.optimization.optimization_config.total_trials}...")
        
        # Ax Bayesian algorithm proposes the optimal mathematical combination to test next
        parameters, trial_index = ax_client.get_next_trial()
        
        # Dynamically inject Ax's proposed numbers destructively directly back into our master python config dictionary
        for param_name, param_value in parameters.items():
            util.set_by_path(cfg, param_name, param_value)

        # Stand up fresh Model architecture parameterized natively via the new config
        if cfg.model.name == "Ensemble_MLP":
            model = Ensemble_MLP(cfg.model)
        elif cfg.model.name == "Autoencoder":
            model = AutoEncoder(cfg.model)
        else:
            raise NotImplementedError(f"Model {cfg.model.name} not implemented in training loop.")
        
        model_summary = util.get_model_summary(model)
        logging.info(model_summary)

        # Connect tracking to the cloud specifically distinguishing optimization epochs
        if cfg.logging.use_wandb and cfg.optimization.wb_logging:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M')
            wandb.login(key=cfg.logging.api_key)
            additional = cfg.logging.run_name_additional
            if additional != 'None' and additional is not None:
                run_name = f"{model.get_name()}_{additional}_optimizationTrial{i + 1}_{timestamp}"
            else:
                run_name = f"{model.get_name()}_optimizationTrial{i + 1}_{timestamp}"
            wb_run = wandb.init(
                entity=cfg.logging.team_name,
                project=cfg.logging.project_name,
                name=run_name,
                config={
                    "architecture": model.get_name(),
                    "dataset": "UjiIndoorLoc",
                    "training settings": {**dict(cfg.training)},
                    "model settings": {**dict(cfg.model)},
                    "model architecture": model_summary,
                    "dataaugmentation settings": {**dict(cfg.dataset.data_augmentation)},
                }
            )
        else:
            wb_run = None

        # Build Trainer Contexts
        evaluator = Evaluator(cfg, eval_dataloader, model, device, wb_run)
        trainer = Trainer(cfg, train_dataloader, model, evaluator, device, wb_run)
        
        # Drive forward pass epochs to convergence
        trainer.train()
        
        # Force evaluation pass mapping performance outputs for the Bayesian client
        metrics = evaluator.eval(return_metrics=True)
        raw_data = {
            "classification_accuracy": metrics['accuracy_evaluation'].item(),
            "regression_mse": metrics['mse'].item()
        }
        print(raw_data)
        
        # Conclude Ax iteration by feeding back empirical results (informs the NEXT trial's logic search)
        ax_client.complete_trial(trial_index=trial_index, raw_data=raw_data)

        # 5. Purge memory between structural network resets
        del model
        del trainer
        del evaluator
        wb_run.finish() if wb_run is not None else None
        del wb_run
        gc.collect()

    # 6. Retrieve Mathematical Optimum
    # Extracts the absolute mathematical peak from the parameter distributions
    pareto_results = ax_client.get_pareto_optimal_parameters()
    best_params = list(pareto_results.values())[0]
    
    print("\n--- Optimization complete ---")
    print(f"Best parameters: {best_params[0]}")
    print(f"Expected performance: {best_params[1]}")
    print("\n\n")
    print("Training & evaluating best model on test set")

    # 7. Final Sanity Benchmark execution
    # Permanently overwrite CFG parameters dynamically matching our best discoveries
    for param_name, param_value in best_params[0].items():
        util.set_by_path(cfg, param_name, param_value)
    
    # Commit optimal hyperparameter block natively strictly writing out to local disk
    util.save_best_parameters(cfg, filename="best_parameters.yaml")

    # Re-initialize Peak Matrix network layout
    if cfg.model.name == "Ensemble_MLP":
        model = Ensemble_MLP(cfg.model)
    elif cfg.model.name == "Autoencoder":
        model = AutoEncoder(cfg.model)
    else:
        raise NotImplementedError(f"Model {cfg.model.name} not implemented in training loop.")

    # Boot one final telemetry cloud tracer 
    if cfg.logging.use_wandb:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M')
        wandb.login(key=cfg.logging.api_key)
        additional = cfg.logging.run_name_additional
        if additional != 'None' and additional is not None:
            run_name = f"{model.get_name()}_{additional}_bestResult_{timestamp}"
        else:
            run_name = f"{model.get_name()}_bestResult_{timestamp}"
        wb_run = wandb.init(
            entity=cfg.logging.team_name,
            project=cfg.logging.project_name,
            name=run_name,
            config={
                "architecture": model.get_name(),
                "dataset": "UjiIndoorLoc",
                "training settings": {**dict(cfg.training)},
                "model settings": {**dict(cfg.model)},
                "model architecture": model_summary,
                "dataaugmentation settings": {**dict(cfg.dataset.data_augmentation)},
            }
        )
    else:
        wb_run = None
    
    evaluator = Evaluator(cfg, eval_dataloader, model, device, wb_run)
    trainer = Trainer(cfg, train_dataloader, model, evaluator, device, wb_run)
    
    trainer.train()

    # Assess Peak Neural Net blind accurately against strictly segregated Test Set data
    test_evaluator = Evaluator(cfg, test_dataloader, trainer.get_trained_model(), device, wb_run)
    logging.info("Running Final Evaluation on pure test set")
    test_evaluator.eval()

In [42]:
if __name__ == '__main__':
    if CONFIG.optimization.active:
        optimize()
    else:
        main()

C:\Users\nicol\AppData\Local\Temp\ipykernel_7624\1966084414.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['CLASSID'] = data['BUILDINGID'].astype(str) + '_' + data['SPACEID'].astype(str)
C:\Users\nicol\AppData\Local\Temp\ipykernel_7624\1966084414.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['CLASSID'] = data['BUILDINGID'].astype(str) + '_' + data['SPACEID'].astype(str)
C:\Users\nicol\AppData\Local\Temp\ipykernel_7624\1966084414.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually th

└─ Ensemble_MLP: 334983 trainable parameters
  └─ Classifier: 167685 trainable parameters
    └─ Sequential: 167685 trainable parameters
      └─ Linear: 133376 trainable parameters
      └─ BatchNorm1d: 512 trainable parameters
      └─ LeakyReLU: 0 trainable parameters
      └─ Dropout: 0 trainable parameters
      └─ Linear: 32896 trainable parameters
      └─ BatchNorm1d: 256 trainable parameters
      └─ LeakyReLU: 0 trainable parameters
      └─ Linear: 645 trainable parameters
  └─ Regressor: 167298 trainable parameters
    └─ Sequential: 167298 trainable parameters
      └─ Linear: 133376 trainable parameters
      └─ BatchNorm1d: 512 trainable parameters
      └─ LeakyReLU: 0 trainable parameters
      └─ Dropout: 0 trainable parameters
      └─ Linear: 32896 trainable parameters
      └─ BatchNorm1d: 256 trainable parameters
      └─ LeakyReLU: 0 trainable parameters
      └─ Linear: 258 trainable parameters
Total Trainable Parameters: 334983



--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\nicol\AppData\Local\Python\pythoncore-3.13-64\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\nicol\AppData\Local\Python\pythoncore-3.13-64\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode characters in position 554-555: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Projekte\INTAR_RSSI_localization\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Projekte\INTAR_RSSI_localization\.venv\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


classification_loss_training,█▆▅▄▄▃▃▃▂▃▂▂▂▂▂▂▂▂▂▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
coordinates_mse_training,█▆▃▃▂▁▂▁▁▂▂▁▁▁▁▁▁▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▂▁▂▁▁▁▁
epoch_duration,▁█
floor_accuracy_training,▁▂▃▄▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████████████████
learning_rate,████████████████████▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▁▁▁
total_loss_training,█▄▄▅▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▃▁▁▂▂▁▁▁▁▁▁
classification_loss_training,0.0048
coordinates_mse_training,0.00345
epoch_duration,19.13551
floor_accuracy_training,0.98317
learning_rate,0.0003


Epoch [1/75]
Epoch [2/75]


KeyboardInterrupt: 